# Evaluation Notebook - QA Model V3

Notebook dédié à l'évaluation et à l'optimisation des scores de `bidaf_model_v3_best.pt` (sans ré-entraînement).


## 1) Imports et Device (Apple MPS)


In [1]:
import os
import re
import json
import math
import string
from collections import Counter

import torch
import numpy as np

from models import QAModelV3

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


Device: mps


## 2) Configuration


In [2]:
data_path = "data/"
train_path = os.path.join(data_path, "train-v2.0.json")
dev_path = os.path.join(data_path, "dev-v2.0.json")
ckpt_path = "bidaf_model_v3_best.pt"

MAX_CONTEXT_LEN = 220
MAX_QUESTION_LEN = 40
MIN_FREQ = 2

EMBED_DIM = 200
HIDDEN_SIZE = 128

EVAL_MAX_EXAMPLES = 3000  # None = full dev

BASE_THRESHOLD = 0.50
BASE_MAX_ANSWER_LEN = 30
TOPK = 20
LENGTH_PENALTY = 0.02


## 3) Chargement SQuAD v2 (pipeline V3)


In [3]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")

def tokenize_advanced(text):
    return TOKEN_PATTERN.findall(text.lower())

def char_to_token_span(context, context_tokens, answer_start, answer_text):
    lowered = context.lower()
    offsets = []
    cursor = 0
    for tok in context_tokens:
        idx = lowered.find(tok, cursor)
        if idx == -1:
            idx = lowered.find(tok)
            if idx == -1:
                return None, None
        offsets.append((idx, idx + len(tok)))
        cursor = idx + len(tok)

    ans_start = answer_start
    ans_end = answer_start + len(answer_text)
    s_tok = e_tok = None
    for i, (s, e) in enumerate(offsets):
        if s <= ans_start < e:
            s_tok = i
        if s < ans_end <= e:
            e_tok = i
            break
    return s_tok, e_tok

def load_squad_v2(path, max_examples=None):
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    examples = []
    for article in squad["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            context_tokens = tokenize_advanced(context)

            for qa in paragraph["qas"]:
                question_tokens = tokenize_advanced(qa["question"])
                is_impossible = qa.get("is_impossible", False)

                if is_impossible:
                    examples.append({
                        "context_tokens": context_tokens,
                        "question_tokens": question_tokens,
                        "start": 0,
                        "end": 0,
                        "has_answer": 0,
                        "raw_answer": "",
                    })
                else:
                    answers = qa.get("answers", [])
                    if not answers:
                        continue
                    ans = answers[0]
                    s_tok, e_tok = char_to_token_span(context, context_tokens, ans["answer_start"], ans["text"].lower())
                    if s_tok is None or e_tok is None:
                        continue
                    examples.append({
                        "context_tokens": context_tokens,
                        "question_tokens": question_tokens,
                        "start": s_tok + 1,
                        "end": e_tok + 1,
                        "has_answer": 1,
                        "raw_answer": ans["text"],
                    })

                if max_examples is not None and len(examples) >= max_examples:
                    return examples

    return examples

train_examples = load_squad_v2(train_path, None)
dev_examples = load_squad_v2(dev_path, EVAL_MAX_EXAMPLES)
print("Train examples:", len(train_examples))
print("Dev examples:", len(dev_examples))


Train examples: 130309
Dev examples: 3000


## 4) Vocab + Encodage


In [4]:
def build_vocab(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex["context_tokens"])
        counter.update(ex["question_tokens"])
    vocab = {"<PAD>": 0, "<UNK>": 1, "<CLS>": 2}
    for tok, freq in counter.items():
        if freq >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab

def encode(tokens, vocab_obj, max_len, add_cls=False):
    if add_cls:
        tokens = ["<CLS>"] + tokens
    ids = [vocab_obj.get(t, vocab_obj["<UNK>"]) for t in tokens][:max_len]
    return ids + [vocab_obj["<PAD>"]] * (max_len - len(ids))

vocab = build_vocab(train_examples, MIN_FREQ)
VOCAB_SIZE = len(vocab)
print("VOCAB_SIZE:", VOCAB_SIZE)


VOCAB_SIZE: 80404


## 5) Charger le Modèle V3


In [5]:
assert os.path.exists(ckpt_path), f"Checkpoint introuvable: {ckpt_path}"

model_v3 = QAModelV3(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)
model_v3.load_state_dict(torch.load(ckpt_path, map_location=device))
model_v3.eval()
print("Checkpoint chargé:", ckpt_path)


Checkpoint chargé: bidaf_model_v3_best.pt


## 6) Métriques + Décodage


In [8]:
def normalize_answer(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def compute_em(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))

def compute_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if not p and not t:
        return 1.0
    if not p or not t:
        return 0.0
    common = Counter(p) & Counter(t)
    n = sum(common.values())
    if n == 0:
        return 0.0
    precision = n / len(p)
    recall = n / len(t)
    return 2 * precision * recall / (precision + recall)

def clean_prediction(prediction):
    prediction = re.sub(r"^[\s\(\)\[\],;:\.""]+", "", prediction)
    prediction = re.sub(r"[\s\(\)\[\],;:\.""]+$", "", prediction)
    prediction = re.sub(r'\s+', ' ', prediction).strip()
    return prediction

@torch.no_grad()
def forward_logits(model, context_tokens, question_tokens):
    ctx_ids = torch.LongTensor([encode(context_tokens, vocab, MAX_CONTEXT_LEN, add_cls=True)]).to(device)
    qst_ids = torch.LongTensor([encode(question_tokens, vocab, MAX_QUESTION_LEN, add_cls=False)]).to(device)
    s_logits, e_logits, na_logit = model(ctx_ids, qst_ids)
    return s_logits[0], e_logits[0], torch.sigmoid(na_logit[0]).item()

@torch.no_grad()
def decode_joint_greedy(start_logits, end_logits, max_answer_len):
    best_score = -1e30
    best_s, best_e = 0, 0
    L = start_logits.size(0)
    for s in range(L):
        e_max = min(L - 1, s + max_answer_len - 1)
        e_rel = torch.argmax(end_logits[s:e_max+1]).item()
        e = s + e_rel
        score = start_logits[s].item() + end_logits[e].item()
        if score > best_score:
            best_score = score
            best_s, best_e = s, e
    return best_s, best_e

@torch.no_grad()
def decode_joint_topk(start_logits, end_logits, max_answer_len=30, topk=20, length_penalty=0.0):
    L = start_logits.size(0)
    k = min(topk, L)
    s_vals, s_idx = torch.topk(start_logits, k=k)
    e_vals, e_idx = torch.topk(end_logits, k=k)

    best_score = -1e30
    best_s, best_e = 0, 0

    for i in range(k):
        s = s_idx[i].item()
        for j in range(k):
            e = e_idx[j].item()
            if e < s:
                continue
            span_len = e - s + 1
            if span_len > max_answer_len:
                continue
            score = s_vals[i].item() + e_vals[j].item() - length_penalty * span_len
            if score > best_score:
                best_score = score
                best_s, best_e = s, e

    return best_s, best_e

def span_to_text(context_tokens, s_idx, e_idx):
    if s_idx == 0 or e_idx == 0:
        return ""
    s_txt = s_idx - 1
    e_txt = min(e_idx - 1, len(context_tokens) - 1)
    if e_txt < s_txt:
        return ""
    return clean_prediction(' '.join(context_tokens[s_txt:e_txt+1]))


## 7) Evaluation Baseline V3 (settings actuels)


In [ ]:
def evaluate_v3(threshold=0.5, max_answer_len=30, decoder='greedy', topk=20, length_penalty=0.0):
    em_total = 0
    f1_total = 0.0
    n = 0

    for ex in dev_examples:
        if ex['start'] >= MAX_CONTEXT_LEN or ex['end'] >= MAX_CONTEXT_LEN:
            continue

        s_logits, e_logits, p_no = forward_logits(model_v3, ex['context_tokens'], ex['question_tokens'])

        if p_no >= threshold:
            pred = ''
        else:
            if decoder == 'topk':
                s_idx, e_idx = decode_joint_topk(s_logits, e_logits, max_answer_len=max_answer_len, topk=topk, length_penalty=length_penalty)
            else:
                s_idx, e_idx = decode_joint_greedy(s_logits, e_logits, max_answer_len=max_answer_len)
            pred = span_to_text(ex['context_tokens'], s_idx, e_idx)

        truth = ex['raw_answer']
        em_total += compute_em(pred, truth)
        f1_total += compute_f1(pred, truth)
        n += 1

    return {
        'n_eval': n,
        'em': 100.0 * em_total / max(n, 1),
        'f1': 100.0 * f1_total / max(n, 1),
    }

baseline = evaluate_v3(threshold=BASE_THRESHOLD, max_answer_len=BASE_MAX_ANSWER_LEN, decoder='greedy')
print('Baseline V3:')
print(f"N={baseline['n_eval']} | EM={baseline['em']:.2f}% | F1={baseline['f1']:.2f}%")


## 8) Sweep du Seuil No-Answer


In [ ]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
rows_threshold = []

for th in thresholds:
    m = evaluate_v3(threshold=th, max_answer_len=BASE_MAX_ANSWER_LEN, decoder='greedy')
    rows_threshold.append((th, m['em'], m['f1']))

print('| Threshold | EM (%) | F1 (%) |')
print('|---:|---:|---:|')
for th, em, f1 in rows_threshold:
    print(f'| {th:.2f} | {em:.2f} | {f1:.2f} |')

best_th = max(rows_threshold, key=lambda x: x[2])
print(f"\nBest threshold by F1: {best_th[0]:.2f} (F1={best_th[2]:.2f}%)")


## 9) Sweep Décodage Top-K + Longueur


In [ ]:
candidate_max_len = [15, 20, 30, 40]
candidate_penalty = [0.00, 0.01, 0.02, 0.05]
best_threshold = max(rows_threshold, key=lambda x: x[2])[0]

rows_decode = []
for ml in candidate_max_len:
    for pen in candidate_penalty:
        m = evaluate_v3(
            threshold=best_threshold,
            max_answer_len=ml,
            decoder='topk',
            topk=TOPK,
            length_penalty=pen,
        )
        rows_decode.append((ml, pen, m['em'], m['f1']))

print('| MaxAnswerLen | LengthPenalty | EM (%) | F1 (%) |')
print('|---:|---:|---:|---:|')
for ml, pen, em, f1 in rows_decode:
    print(f'| {ml} | {pen:.2f} | {em:.2f} | {f1:.2f} |')

best_decode = max(rows_decode, key=lambda x: x[3])
print(f"\nBest decoding by F1: max_len={best_decode[0]}, penalty={best_decode[1]:.2f}, F1={best_decode[3]:.2f}%")


## 10) Tableau Final (Baseline vs Optimisé)


In [ ]:
best_threshold = max(rows_threshold, key=lambda x: x[2])[0]
best_decode = max(rows_decode, key=lambda x: x[3])

optimized = evaluate_v3(
    threshold=best_threshold,
    max_answer_len=best_decode[0],
    decoder='topk',
    topk=TOPK,
    length_penalty=best_decode[1],
)

print('| Configuration | N évalués | EM (%) | F1 (%) |')
print('|---|---:|---:|---:|')
print(f"| Baseline (greedy, th={BASE_THRESHOLD:.2f}, max_len={BASE_MAX_ANSWER_LEN}) | {baseline['n_eval']} | {baseline['em']:.2f} | {baseline['f1']:.2f} |")
print(f"| Optimisée (topk={TOPK}, th={best_threshold:.2f}, max_len={best_decode[0]}, pen={best_decode[1]:.2f}) | {optimized['n_eval']} | {optimized['em']:.2f} | {optimized['f1']:.2f} |")

print('\nDelta (optimisé - baseline):')
print(f"EM: {optimized['em'] - baseline['em']:+.2f} points")
print(f"F1: {optimized['f1'] - baseline['f1']:+.2f} points")


## 11) Prochaines Etapes Recommandées


In [ ]:
print("1) Appliquer dans le notebook d'entraînement v3 le threshold optimal trouvé.")
print("2) Garder le décodage top-k en inference par défaut.")
print("3) Lancer un second training avec HIDDEN_SIZE=192 ou 256 + early stopping.")
print("4) Refaire exactement ce notebook d'évaluation et comparer les deltas.")
